# Análise Exploratória dos Dados

## Dicionário de dados

In [1]:
# importar os pacotes necessários
import pandas as pd
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
dicionario = pd.read_csv("../data/raw/dicionario_dados.csv", sep=";")
dicionario

,base,coluna,descricao
0,base_cadastral,id_cliente,Identificador único do cliente
1,base_cadastral,sexo,Sexo do cliente (M ou F)
2,base_cadastral,data_nascimento,Data de nascimento do cliente
3,base_cadastral,qtd_filhos,Quantidade de filhos declarados
4,base_cadastral,qtd_membros_familia,Quantidade total de membros da família
...,...,...,...
64,historico_parcelas,numero_parcela,Número sequencial da parcela dentro do contrato
65,historico_parcelas,data_prevista_pagamento,Data inicialmente prevista para o pagamento da...
66,historico_parcelas,data_real_pagamento,Data real em que o pagamento foi efetuado
67,historico_parcelas,valor_previsto_parcela,Valor original previsto da parcela


## Funções

In [3]:
def agg_hist_contract(hist_contratos, hist_parcelas):
    # unir historico de contratos com parcelas
    hist_total = hist_contratos.merge(
        hist_parcelas, 
        how="left", 
        on=["id_contrato", "id_cliente"]
    )

    hist_total["dias_atraso_parcela"] = (pd.to_datetime(hist_total["data_real_pagamento"]) - pd.to_datetime(hist_total["data_prevista_pagamento"])).dt.days

    # CRIACAO DA VARIÁVEL ALVO DE ACORDO COM O REGULADOR (BACEN)
    hist_total["target"] = np.where(
        hist_total["dias_atraso_parcela"] > 90,
        1,
        0
    )

    features_temp = hist_total.groupby(
        [
            "id_cliente",
            "id_contrato", 
            "tipo_contrato", 
            "status_contrato", 
            "data_decisao", 
            "valor_credito", 
            "valor_bem", 
            "percentual_entrada",
            "qtd_parcelas_planejadas",
            "tipo_pagamento",
            "finalidade_emprestimo",
            "tipo_cliente",
            "faixa_rendimento",
            "tipo_portfolio",
            "tipo_produto",
            "categoria_bem",
            "combinacao_produto",
            "setor_vendedor",
            "dia_semana_solicitacao",
            "hora_solicitacao",
            "flag_ultima_solicitacao_contrato",
            "flag_ultima_solicitacao_dia",
            "motivo_recusa",
            "acompanhantes_cliente",
            "flag_seguro_contratado"
        ]
    ).agg(
    num_max_parcelas=("numero_parcela", "max"),
    valor_total_a_pagar=("valor_parcela", "sum"),
    valor_medio_pago_contrato=("valor_pago_parcela", "mean"),
    valor_max_pago_contrato=("valor_pago_parcela", "max"),
    target=("target", "max"),
    ).reset_index()

    features_temp = features_temp.sort_values(by=["id_cliente", "data_decisao"])
    return features_temp

## Criação das Métricas Históricas

In [4]:
# importar as tabelas
cadastral = pd.read_parquet('../data/raw/base_cadastral.parquet', engine="fastparquet")
submissao = pd.read_parquet('../data/raw/base_submissao.parquet', engine="fastparquet") 
hist_contratos = pd.read_parquet('../data/raw/historico_emprestimos.parquet', engine="fastparquet")
hist_parcelas = pd.read_parquet('../data/raw/historico_parcelas.parquet', engine="fastparquet")

In [5]:
submissao.head()

,id_cliente,data_solicitacao,dia_semana_solicitacao,hora_solicitacao,tipo_contrato,valor_credito,valor_bem,valor_parcela
0,100023,2025-02-24,MONDAY,12,Cash loans,544491.0,454500.0,17563.5
1,100031,2025-02-17,MONDAY,9,Cash loans,979992.0,702000.0,27076.5
2,100056,2025-02-20,THURSDAY,10,Cash loans,1506816.0,1350000.0,49927.5
3,100069,2025-02-10,MONDAY,11,Cash loans,640458.0,517500.0,27265.5
4,100085,2025-02-19,WEDNESDAY,12,Cash loans,755190.0,675000.0,28894.5


### 1. Agregar Histórico por Contrato

In [6]:
df_hist_contract = agg_hist_contract(hist_contratos, hist_parcelas)
df_hist_contract

,id_cliente,id_contrato,tipo_contrato,status_contrato,data_decisao,valor_credito,valor_bem,percentual_entrada,qtd_parcelas_planejadas,tipo_pagamento,...,flag_ultima_solicitacao_contrato,flag_ultima_solicitacao_dia,motivo_recusa,acompanhantes_cliente,flag_seguro_contratado,num_max_parcelas,valor_total_a_pagar,valor_medio_pago_contrato,valor_max_pago_contrato,target
0,100023,1131053,Consumer loans,Approved,2018-07-07,76680.0,78705.0,0.101380,10.0,Cash through the bank,...,Y,1,XAP,"Spouse, partner",1.0,8.0,68153.040,10575.292500,24968.430,0
1,100023,2411919,Consumer loans,Approved,2020-02-01,93145.5,91282.5,0.095959,16.0,Cash through the bank,...,Y,1,XAP,Other_B,0.0,14.0,111888.000,9046.992857,22761.900,0
2,100067,1109767,Consumer loans,Approved,2017-06-05,34821.0,51975.0,0.359447,10.0,XNA,...,Y,1,XAP,Unaccompanied,0.0,10.0,42620.850,4260.937500,4262.085,0
4,100067,1821761,Consumer loans,Approved,2018-07-09,31410.0,53910.0,0.454545,10.0,Cash through the bank,...,Y,1,XAP,Other_B,0.0,10.0,42529.050,4246.749000,4252.905,0
3,100067,1690355,Consumer loans,Approved,2024-10-13,68346.0,68346.0,0.000000,10.0,Cash through the bank,...,Y,1,XAP,Family,0.0,4.0,31572.900,7893.225000,7893.225,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48218,456236,2436406,Cash loans,Approved,2020-10-17,359923.5,324000.0,0.000000,24.0,Cash through the bank,...,Y,1,XAP,Unaccompanied,1.0,9.0,253267.245,42507.290455,283385.835,0
48219,456239,1979295,Consumer loans,Approved,2017-02-08,16839.0,18112.5,0.105887,4.0,Cash through the bank,...,Y,1,XAP,Unaccompanied,1.0,4.0,19189.800,4797.101250,4797.450,0
48220,456242,1128311,Consumer loans,Approved,2023-02-21,173380.5,141426.0,0.000000,12.0,Cash through the bank,...,Y,1,XAP,Unaccompanied,0.0,12.0,216696.600,18051.540000,18058.050,0
48221,456242,2186608,Consumer loans,Approved,2024-06-24,78565.5,72211.5,0.000000,10.0,Non-cash from your account,...,Y,1,XAP,Unaccompanied,1.0,8.0,74927.160,9365.895000,9365.895,0


### 2. Criar Janela Temporal Agregando as Métricas por Janela de "data_decisao"

In [12]:
# Colunas que pegam o valor do último contrato da janela
COLUNAS_ULTIMO_VALOR = [
    "tipo_contrato",
    "status_contrato",
    "tipo_pagamento",
    "finalidade_emprestimo",
    "tipo_cliente",
    "faixa_rendimento",
    "tipo_portfolio",
    "tipo_produto",
    "categoria_bem",
    "combinacao_produto",
    "setor_vendedor",
    "motivo_recusa",
    "acompanhantes_cliente",
    "flag_seguro_contratado",
    "flag_ultima_solicitacao_contrato",
    "flag_ultima_solicitacao_dia",
    "dia_semana_solicitacao",
    "hora_solicitacao",
]


def rolling_features_por_cliente(grupo):
    rows = []

    for i in range(len(grupo)):
        janela = grupo.iloc[[0]] if i == 0 else grupo.iloc[0:i]

        row = {}

        # grupo.name retorna o valor do id_cliente do grupo atual
        row["id_cliente"]   = grupo.name
        row["id_contrato"]  = grupo.iloc[i]["id_contrato"]
        row["data_decisao"] = grupo.iloc[i]["data_decisao"]
        row["target"]       = grupo.iloc[i]["target"]

        row["hist_qtd_contratos"]                = len(janela)
        row["hist_target_max"]                   = janela["target"].max()
        row["hist_target_mean"]                  = janela["target"].mean()
        row["hist_valor_credito_medio"]          = janela["valor_credito"].mean()
        row["hist_valor_credito_max"]            = janela["valor_credito"].max()
        row["hist_valor_credito_min"]            = janela["valor_credito"].min()
        row["hist_valor_bem_medio"]              = janela["valor_bem"].mean()
        row["hist_valor_bem_max"]                = janela["valor_bem"].max()
        row["hist_valor_bem_min"]                = janela["valor_bem"].min()
        row["hist_percentual_entrada_medio"]     = janela["percentual_entrada"].mean()
        row["hist_qtd_parcelas_planejadas_max"]  = janela["qtd_parcelas_planejadas"].max()
        row["hist_parcelas_max"]                 = janela["num_max_parcelas"].max()
        row["hist_parcelas_medio"]               = janela["num_max_parcelas"].mean()
        row["hist_valor_total_pago_sum"]         = janela["valor_total_a_pagar"].sum()
        row["hist_valor_total_pago_medio"]       = janela["valor_total_a_pagar"].mean()
        row["hist_pago_medio"]                   = janela["valor_medio_pago_contrato"].mean()
        row["hist_pago_max"]                     = janela["valor_max_pago_contrato"].max()

        ultimo = janela.iloc[-1]
        for col in COLUNAS_ULTIMO_VALOR:
            row[f"hist_ultimo_{col}"] = ultimo[col]

        rows.append(row)

    return pd.DataFrame(rows)


df_window = (
    df_hist_contract
    .sort_values(["id_cliente", "data_decisao"])
    .reset_index(drop=True)
    .groupby("id_cliente", group_keys=False)
    .apply(rolling_features_por_cliente)
    .reset_index(drop=True)
)

"""
Lógica da análise realizada acima

Contrato  │  Numéricas (ex: valor_credito)   │  Categóricas (ex: tipo_contrato)
──────────────────────────────────────────────────────────────────────────────
   1      │  mean/max/min do próprio          │  valor do próprio contrato
   2      │  mean/max/min do [1]              │  valor do contrato 1
   3      │  mean/max/min do [1, 2]           │  valor do contrato 2
   4      │  mean/max/min do [1, 2, 3]        │  valor do contrato 3
   5      │  mean/max/min do [1, 2, 3, 4]     │  valor do contrato 4
"""

'\nLógica da análise realizada acima\n\nContrato  │  Numéricas (ex: valor_credito)   │  Categóricas (ex: tipo_contrato)\n──────────────────────────────────────────────────────────────────────────────\n   1      │  mean/max/min do próprio          │  valor do próprio contrato\n   2      │  mean/max/min do [1]              │  valor do contrato 1\n   3      │  mean/max/min do [1, 2]           │  valor do contrato 2\n   4      │  mean/max/min do [1, 2, 3]        │  valor do contrato 3\n   5      │  mean/max/min do [1, 2, 3, 4]     │  valor do contrato 4\n'

In [13]:
df_window.info()

<class 'pandas.DataFrame'>
RangeIndex: 48223 entries, 0 to 48222
Data columns (total 39 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id_cliente                                    48223 non-null  int64  
 1   id_contrato                                   48223 non-null  int64  
 2   data_decisao                                  48223 non-null  str    
 3   target                                        48223 non-null  int64  
 4   hist_qtd_contratos                            48223 non-null  int64  
 5   hist_target_max                               48223 non-null  int64  
 6   hist_target_mean                              48223 non-null  float64
 7   hist_valor_credito_medio                      48223 non-null  float64
 8   hist_valor_credito_max                        48223 non-null  float64
 9   hist_valor_credito_min                        48223 non-null  float64
 1

In [14]:
df_window

,id_cliente,id_contrato,data_decisao,target,hist_qtd_contratos,hist_target_max,hist_target_mean,hist_valor_credito_medio,hist_valor_credito_max,hist_valor_credito_min,...,hist_ultimo_categoria_bem,hist_ultimo_combinacao_produto,hist_ultimo_setor_vendedor,hist_ultimo_motivo_recusa,hist_ultimo_acompanhantes_cliente,hist_ultimo_flag_seguro_contratado,hist_ultimo_flag_ultima_solicitacao_contrato,hist_ultimo_flag_ultima_solicitacao_dia,hist_ultimo_dia_semana_solicitacao,hist_ultimo_hora_solicitacao
0,100023,1131053,2018-07-07,0,1,0,0.0,76680.0,76680.0,76680.0,...,Consumer Electronics,POS household without interest,Consumer electronics,XAP,"Spouse, partner",1.0,Y,1,SATURDAY,10
1,100023,2411919,2020-02-01,0,1,0,0.0,76680.0,76680.0,76680.0,...,Consumer Electronics,POS household without interest,Consumer electronics,XAP,"Spouse, partner",1.0,Y,1,SATURDAY,10
2,100067,1109767,2017-06-05,0,1,0,0.0,34821.0,34821.0,34821.0,...,Mobile,POS mobile with interest,Connectivity,XAP,Unaccompanied,0.0,Y,1,MONDAY,12
3,100067,1821761,2018-07-09,0,1,0,0.0,34821.0,34821.0,34821.0,...,Mobile,POS mobile with interest,Connectivity,XAP,Unaccompanied,0.0,Y,1,MONDAY,12
4,100067,1690355,2024-10-13,0,2,0,0.0,33115.5,34821.0,31410.0,...,Mobile,POS mobile with interest,Connectivity,XAP,Other_B,0.0,Y,1,MONDAY,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48218,456236,2436406,2020-10-17,0,1,0,0.0,292455.0,292455.0,292455.0,...,XNA,Cash Street: middle,XNA,XAP,Family,1.0,Y,1,WEDNESDAY,11
48219,456239,1979295,2017-02-08,0,1,0,0.0,16839.0,16839.0,16839.0,...,Mobile,POS mobile with interest,Connectivity,XAP,Unaccompanied,1.0,Y,1,WEDNESDAY,11
48220,456242,1128311,2023-02-21,0,1,0,0.0,173380.5,173380.5,173380.5,...,Audio/Video,POS household with interest,Consumer electronics,XAP,Unaccompanied,0.0,Y,1,TUESDAY,14
48221,456242,2186608,2024-06-24,0,1,0,0.0,173380.5,173380.5,173380.5,...,Audio/Video,POS household with interest,Consumer electronics,XAP,Unaccompanied,0.0,Y,1,TUESDAY,14
